# 03 - Calibrate ResNet Threshold

Step 03 of the TamIA pipeline in Colab format. Scores validation images with DINO top-k candidates plus the ResNet ensemble and writes the calibrated threshold.

Run this notebook with a GPU runtime. The code executes the same root script used by the TamIA Slurm job.

In [ ]:
#@title Colab setup
from pathlib import Path
import os
import subprocess
import sys

# Edit these if needed.
REPO_URL = "https://github.com/moradBMH/INF8225_Projet.git"
GIT_REF = "clean-structure"
PROJECT_DIR = Path("/content/INF8225_Projet")

# If your Drive folder is not auto-detected by colab/setup.py, set it explicitly,
# for example: "/content/drive/MyDrive/Projet_Medsam".
DRIVE_FOLDER = None

# Keep INSTALL_DEPS=True on the first notebook in a fresh Colab runtime.
# You can set it to False for later notebooks in the same runtime.
INSTALL_DEPS = True
REINSTALL_DEPS = False

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--branch", GIT_REF, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", GIT_REF], check=False)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", GIT_REF], check=False)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=False)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from colab.setup import setup
setup(drive_folder=DRIVE_FOLDER, install=INSTALL_DEPS, reinstall=REINSTALL_DEPS)
print("cwd:", Path.cwd())


In [ ]:
#@title Verify shared assets
from pathlib import Path
from msd_recall_strategy import get_resnet_checkpoint_dir

checkpoint_dir = get_resnet_checkpoint_dir()
required = [
    Path("data/MSD_pancreas/train.json"),
    Path("data/MSD_pancreas/val.json"),
    Path("data/MSD_pancreas/test.json"),
    Path("work_dirs/tumor_config_v3/best_coco_bbox_mAP_epoch_25.pth"),
    Path("work_dirs/tumor_config_v3/tumor_config_v3.py"),
]
missing = [p for p in required if not p.exists()]
for p in required:
    print(("OK     " if p.exists() else "MISSING"), p)
assert not missing, f"Missing required files: {missing}"

required += [checkpoint_dir / f"resnet_fold_{i}.pth" for i in range(1, 6)]
missing = [p for p in required if not p.exists()]
for p in required:
    print(("OK     " if p.exists() else "MISSING"), p)
assert not missing, f"Missing required files: {missing}"


In [ ]:
#@title Run pipeline step
import subprocess
import sys
subprocess.run([sys.executable, "-u", "-m", "msd_implementation.pipelines.resnet18_recall.calibrate_threshold"], check=True)


In [ ]:
#@title Inspect threshold artifacts
from pathlib import Path
for p in [Path("outputs/msd_implementation/resnet18_recall/metrics/optimal_threshold_resnet18.txt"), Path("outputs/msd_implementation/resnet18_recall/metrics/optimal_threshold_resnet18.json"), Path("outputs/msd_implementation/resnet18_recall/metrics/calibration_threshold_multi_candidate.csv"), Path("outputs/msd_implementation/resnet18_recall/metrics/threshold_sweep_multi_candidate.csv")]:
    print(("OK     " if p.exists() else "MISSING"), p)
assert Path("outputs/msd_implementation/resnet18_recall/metrics/optimal_threshold_resnet18.txt").exists(), "outputs/msd_implementation/resnet18_recall/metrics/optimal_threshold_resnet18.txt missing"
print("threshold:", Path("outputs/msd_implementation/resnet18_recall/metrics/optimal_threshold_resnet18.txt").read_text().strip())
